## **1.Setting**

In [1]:
# GPU 상태 확인 (Jupyter 커널 또는 다른 Python 프로세스를 강제 종료하지 않습니다)
import os
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 여유: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")
else:
    print("CUDA GPU가 필요합니다. RunPod GPU 런타임을 확인하세요.")
print("이전에 같은 커널에서 모델을 로드했다면 Kernel Restart 후 처음부터 실행하세요.")

GPU: NVIDIA A100-SXM4-80GB
GPU 여유: 78.84 / 79.25 GB
이전에 같은 커널에서 모델을 로드했다면 Kernel Restart 후 처음부터 실행하세요.


In [2]:
# Gradio 버전 충돌 최소화: vLLM/Transformers/torch는 건드리지 않음
%pip install -q \
    "vllm==0.11.1" \
    "numpy==2.2.6" \
    "pandas>=2.2,<3.0" \
    "datasets>=3.6,<4.0" \
    "huggingface_hub>=0.30,<1.0" \
    "transformers>=4.50,<4.58" \
    "peft>=0.15,<0.18" \
    "openai>=2.0,<3.0" \
    "tqdm>=4.66,<5.0" \
    "gradio>=4.44,<6"
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# 이미 설치된 Hugging Face Hub 버전은 유지합니다
# 패키지를 설치한 직후 import 충돌이 나면 Kernel Restart 후 다시 실행하세요

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 필요 라이브러리 및 선택적 Hugging Face 로그인
import os
import json
import re
from pathlib import Path
import pandas as pd
import gradio as gr
from vllm import LLM, SamplingParams
from huggingface_hub import login, HfApi
from getpass import getpass

HF_TOKEN = (os.getenv("HF_TOKEN") or "").strip()
HF_USERNAME = (os.getenv("PILATES_HF_USERNAME") or "").strip()

# 모델 위치를 PILATES_MODEL_REPO(로컬 병합 경로 또는 Hub 저장소 ID)로 지정했다면 토큰은 선택입니다
if not HF_TOKEN and not os.getenv("PILATES_MODEL_REPO") and not HF_USERNAME:
    HF_TOKEN = getpass("Hugging Face 토큰 (비공개 저장소 접근 시 필요): ").strip()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    if not HF_USERNAME:
        HF_USERNAME = HfApi().whoami(token=HF_TOKEN)["name"]
print("Hugging Face 사용자:", HF_USERNAME or "미지정 (모델 경로 직접 지정 필요)")

# DB/시간 기반 함수가 참조하는 데모 기준일
REFERENCE_DATE = "2026-11-02"

Hugging Face 토큰 (비공개 저장소 접근 시 필요):  ········


Hugging Face 사용자: somsom317


## **2. Data & Function Load**

In [4]:
# 데이터 불러오기
# DB 로딩
cwd = Path.cwd()
candidates = [cwd / "database", cwd / "Part6" / "database", cwd, Path("/mnt/data")]
DATA_DIR = next((p for p in candidates if list(p.glob("users*.json"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("database 폴더에 users.json 등 8개 JSON을 넣어주세요.")

def load_json(filename, alternative=None):
    path = DATA_DIR / filename
    if not path.exists() and alternative:
        path = DATA_DIR / alternative
    if not path.exists():
        # 업로드된 users(4).json 등 버전 번호가 다른 파일 지원
        prefix = filename.split("(")[0].removesuffix(".json")
        matches = sorted(DATA_DIR.glob(f"{prefix}*.json"))
        if not matches:
            raise FileNotFoundError(f"{prefix} JSON 파일을 찾을 수 없습니다: {DATA_DIR}")
        path = matches[-1]
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

df_users = load_json("users(1).json", "users.json")
df_memberships = load_json("memberships(2).json", "memberships.json")
df_classes = load_json("classes(3).json", "classes.json")
df_instructors = load_json("instructors(3).json", "instructors.json")
df_user_memberships = load_json("user_memberships(4).json", "user_memberships.json")
df_bookings = load_json("bookings(4).json", "bookings.json")
df_attendance_logs = load_json("attendance_logs(1).json", "attendance_logs.json")
df_policies = load_json("policies(2).json", "policies.json")

required = {"classes": (df_classes, ["day_of_week"]),
            "bookings": (df_bookings, ["class_date", "week_status"])}
for name, (df, cols) in required.items():
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise ValueError(f"{name} 데이터가 이전 버전입니다: {missing}. 최신 JSON으로 교체해 주세요.")
print("DATA_DIR:", DATA_DIR)
for name, df in [("users",df_users),("memberships",df_memberships),
                 ("classes",df_classes),("instructors",df_instructors),
                 ("user_memberships",df_user_memberships),("bookings",df_bookings),
                 ("policies",df_policies)]:
    print(f"{name:18s}: {len(df)} rows")

DATA_DIR: /workspace/database
users             : 50 rows
memberships       : 17 rows
classes           : 100 rows
instructors       : 8 rows
user_memberships  : 81 rows
bookings          : 105 rows
policies          : 15 rows


In [5]:
# 함수 호출하기
# 2-1. 이용권 상품 정보 조회 함수
def get_membership_info(
    sessions=None,
    product_type=None,
    lesson_type=None
):
    membership_info = df_memberships.copy()

    # 전체 횟수 기준 상품 조회
    if sessions is not None:
        membership_info = membership_info[
            (membership_info['total_sessions'] == sessions)
        ]

    # 상품 유형 검색
    if product_type is not None:
        membership_info = membership_info[
            membership_info['product_type'] == product_type
        ]

    # 레슨 유형 검색
    if lesson_type is not None:
        membership_info = membership_info[
            membership_info['lesson_type'] == lesson_type
        ]

    if membership_info.empty:
        return {
            "success": False,
            "membership_count": 0,
            "message": "조건에 해당하는 이용권이 없습니다."
        }

    return {
        "success": True,
        "membership_count": len(membership_info),
        "membership_info": membership_info.to_dict(orient="records")
    }
    
# 2-2. 수업 시간표 조회 함수
def get_class_schedule(
    day_of_week=None,
    lesson_type=None,
    instructor_name=None,
    start_time=None,
    end_time=None,
    time_filter="exact"
):
    # 수업 정보와 강사 정보 병합
    class_info = df_classes.merge(
        df_instructors[['instructor_id', 'name']],
        on='instructor_id',
        how='left'
    ).rename(columns={'name': 'instructor_name'})

    # 요일별 조회
    if day_of_week is not None:
        class_info = class_info[
            class_info['day_of_week'] == day_of_week
        ]

    # 레슨 유형별 조회
    if lesson_type is not None:
        class_info = class_info[
            class_info['lesson_type'] == lesson_type
        ]

    # 강사 이름별 조회
    if instructor_name is not None:
        class_info = class_info[
            class_info['instructor_name'] == instructor_name
        ]

    # 시간 검색 방식 검증 (초과/미만과 이상/이하 구분)
    if time_filter not in ("exact", "after", "at_or_after", "before", "at_or_before", "between"):
        return {
            "success": False,
            "class_count": 0,
            "message": "지원하지 않는 시간 검색 방식입니다."
        }

    if time_filter == "between":
        # 시작 시각이 두 경계 사이인 수업 조회 (양 끝 포함)
        if start_time is None or end_time is None:
            return {
                "success": False,
                "class_count": 0,
                "message": "시간 범위 조회에는 시작 시간과 종료 시간이 모두 필요합니다."
            }

        class_info = class_info[
            (class_info['start_time'] >= start_time) &
            (class_info['start_time'] <= end_time)
        ]

    else:
        # 시작 시각 비교: after/before는 경계 제외
        if start_time is not None:
            if time_filter == "after":
                class_info = class_info[
                    class_info['start_time'] > start_time
                ]
            elif time_filter == "at_or_after":
                class_info = class_info[
                    class_info['start_time'] >= start_time
                ]
            elif time_filter == "before":
                class_info = class_info[
                    class_info['start_time'] < start_time
                ]
            elif time_filter == "at_or_before":
                class_info = class_info[
                    class_info['start_time'] <= start_time
                ]
            else:
                class_info = class_info[
                    class_info['start_time'] == start_time
                ]

        # between 외에는 end_time 이하에 종료되는 수업 조회
        if end_time is not None:
            class_info = class_info[
                class_info['end_time'] <= end_time
            ]

    # 조회 결과가 없는 경우
    if class_info.empty:
        return {
            "success": False,
            "class_count": 0,
            "message": "조건에 해당하는 수업 시간표가 없습니다."
        }

    # 요일 및 시간순 정렬
    day_order = {
        '월요일': 0,
        '화요일': 1,
        '수요일': 2,
        '목요일': 3,
        '금요일': 4
    }

    class_info = class_info.copy()
    class_info['_day_order'] = class_info['day_of_week'].map(day_order)

    class_info = class_info.sort_values(
        ['_day_order', 'start_time', 'instructor_name']
    )

    # 반환할 수업 정보
    columns = [
        'class_id',
        'day_of_week',
        'start_time',
        'end_time',
        'lesson_type',
        'instructor_id',
        'instructor_name',
        'capacity',
        'status'
    ]

    return {
        "success": True,
        "class_count": len(class_info),
        "class_schedule": class_info[columns].to_dict(orient="records")
    }

# 2-3. 예약 가능한 수업 조회 함수
def get_available_classes(
    day_of_week=None,
    lesson_type=None,
    instructor_name=None,
    start_time=None,
    end_time=None,
    time_filter="exact"
):
    # 1. 개발 환경 기준 시점
    now = pd.Timestamp(REFERENCE_DATE)

    # 예약 오픈 기준: 매주 금요일 12시
    first_open = pd.Timestamp("2026-10-30 12:00")

    if now < first_open:
        return {
            "success": False,
            "available_class_count": 0,
            "message": "아직 예약이 오픈되지 않았습니다."
        }

    days_since_friday = (now.weekday() - 4) % 7
    open_date = now.normalize() - pd.Timedelta(
        days=days_since_friday
    )

    if now < open_date + pd.Timedelta(hours=12):
        open_date -= pd.Timedelta(days=7)

    # 예약이 열린 다음 주 월~금 날짜 생성
    monday = open_date + pd.Timedelta(days=3)

    target_dates = pd.DataFrame({
        "class_date": [
            (monday + pd.Timedelta(days=i)).strftime("%Y-%m-%d")
            for i in range(5)
        ],
        "day_of_week": [
            "월요일", "화요일", "수요일", "목요일", "금요일"
        ]
    })

    # 2. 수업과 강사 정보 병합
    class_info = df_classes.merge(
        df_instructors[["instructor_id", "name"]],
        on="instructor_id",
        how="left"
    ).rename(columns={"name": "instructor_name"})

    # 운영 중인 수업만 조회
    class_info = class_info[
        class_info["status"] == "ACTIVE"
    ].copy()

    # 정규 시간표와 예약 대상 날짜 연결
    class_info = class_info.merge(
        target_dates,
        on="day_of_week",
        how="inner"
    )

    # 3. 검색 조건 적용
    if day_of_week is not None:
        class_info = class_info[
            class_info["day_of_week"] == day_of_week
        ]

    if lesson_type is not None:
        class_info = class_info[
            class_info["lesson_type"] == lesson_type
        ]

    if instructor_name is not None:
        class_info = class_info[
            class_info["instructor_name"] == instructor_name
        ]

    # 시간 검색 방식 검증 (초과/미만과 이상/이하 구분)
    if time_filter not in ("exact", "after", "at_or_after", "before", "at_or_before", "between"):
        return {
            "success": False,
            "available_class_count": 0,
            "message": "지원하지 않는 시간 검색 방식입니다."
        }

    # 4. 시작 시각 조건 적용 (between은 양 끝 포함)
    if time_filter == "between":
        if start_time is None or end_time is None:
            return {
                "success": False,
                "available_class_count": 0,
                "message": "시간 범위 조회에는 두 시간이 필요합니다."
            }

        class_info = class_info[
            (class_info["start_time"] >= start_time) &
            (class_info["start_time"] <= end_time)
        ]

    else:
        if start_time is not None:
            if time_filter == "after":
                class_info = class_info[
                    class_info["start_time"] > start_time
                ]
            elif time_filter == "at_or_after":
                class_info = class_info[
                    class_info["start_time"] >= start_time
                ]
            elif time_filter == "before":
                class_info = class_info[
                    class_info["start_time"] < start_time
                ]
            elif time_filter == "at_or_before":
                class_info = class_info[
                    class_info["start_time"] <= start_time
                ]
            else:
                class_info = class_info[
                    class_info["start_time"] == start_time
                ]

        if end_time is not None:
            class_info = class_info[
                class_info["end_time"] <= end_time
            ]

    # 5. 해당 주차의 활성 예약 내역
    active_bookings = df_bookings[
        df_bookings["week_status"] == "ACTIVE"
    ].copy()

    active_bookings["class_date"] = (
        pd.to_datetime(active_bookings["class_date"])
        .dt.strftime("%Y-%m-%d")
    )

    active_bookings = active_bookings[
        active_bookings["class_date"].isin(
            target_dates["class_date"]
        )
    ]

    # 6. 확정 예약 인원 계산
    confirmed = active_bookings[
        active_bookings["status"] == "CONFIRMED"
    ]

    booking_counts = (
        confirmed.groupby(["class_id", "class_date"])
        .size()
        .reset_index(name="booked_count")
    )

    # 예약이 없는 수업도 포함하도록 LEFT JOIN
    class_info = class_info.merge(
        booking_counts,
        on=["class_id", "class_date"],
        how="left"
    )

    class_info["booked_count"] = (
        class_info["booked_count"].fillna(0).astype(int)
    )

    # 7. 잔여석 계산
    class_info["remaining_seats"] = (
        class_info["capacity"] - class_info["booked_count"]
    ).clip(lower=0)

    # 잔여석이 있는 수업만 반환
    class_info = class_info[
        class_info["remaining_seats"] > 0
    ].copy()

    # 8. 날짜 및 시간순 정렬
    class_info = class_info.sort_values(
        ["class_date", "start_time", "instructor_name"]
    )

    columns = [
        "class_id",
        "class_date",
        "day_of_week",
        "start_time",
        "end_time",
        "lesson_type",
        "instructor_name",
        "capacity",
        "booked_count",
        "remaining_seats"
    ]

    return {
        "success": True,
        "week_start_date": monday.strftime("%Y-%m-%d"),
        "available_class_count": len(class_info),
        "available_classes": class_info[columns].to_dict(
            orient="records"
        ),
        "message": (
            "예약 가능한 수업 조회가 완료되었습니다."
            if not class_info.empty
            else "조건에 해당하는 예약 가능한 수업이 없습니다."
        )
    }

# 2-4. 내 보유 이용권 조회 함수
# 실제 서비스에서는 CURRENT_USER_ID 대신 인증된 세션의 user_id 사용

def get_my_memberships(user_id=None):
    if user_id is None:
        return {"success": False, "message": "로그인 회원을 선택해주세요."}
    user = df_users.loc[df_users["user_id"] == user_id, ["user_id", "name", "status"]]
    if user.empty:
        return {"success": False, "membership_count": 0, "message": "로그인 회원 정보를 찾을 수 없습니다."}

    info = df_user_memberships.loc[df_user_memberships["user_id"] == user_id].merge(
        df_memberships[["membership_id", "membership_name", "product_type", "lesson_type", "components", "total_sessions"]],
        on="membership_id", how="left"
    ).copy()
    today = pd.Timestamp(REFERENCE_DATE).normalize()
    info["is_usable"] = (
        info["status"].eq("ACTIVE") &
        pd.to_datetime(info["start_date"], errors="coerce").le(today) &
        pd.to_datetime(info["expires_at"], errors="coerce").ge(today) &
        pd.to_numeric(info["remaining_sessions"], errors="coerce").gt(0)
    )
    columns = ["user_membership_id", "membership_id", "membership_name",
               "product_type", "lesson_type", "components", "total_sessions",
               "start_date", "expires_at", "remaining_sessions",
               "remaining_sessions_by_type", "status", "is_usable"]
    records = info[columns].astype(object).where(pd.notna(info[columns]), None).to_dict("records")
    return {
        "success": True,
        "user_id": user_id,
        "user_name": user.iloc[0]["name"],
        "reference_date": today.strftime("%Y-%m-%d"),
        "membership_count": len(records),
        "user_memberships": records,
        "message": "내 이용권 조회 완료" if records else "보유 이용권이 없습니다."
    }

# 2-5. 내 예약 내역 조회 함수
def get_my_bookings(status=None, week_status=None, day_of_week=None, user_id=None):
    if user_id is None:
        return {"success": False, "message": "로그인 회원을 선택해주세요."}
    user = df_users.loc[df_users["user_id"] == user_id, ["user_id", "name"]]
    if user.empty:
        return {"success": False, "booking_count": 0,
                "message": "로그인 회원 정보를 찾을 수 없습니다."}

    info = df_bookings.loc[df_bookings["user_id"] == user_id].copy()
    info = info.merge(
        df_classes[["class_id", "day_of_week", "start_time", "end_time",
                    "lesson_type", "instructor_id"]],
        on="class_id", how="left"
    ).merge(
        df_instructors[["instructor_id", "name"]].rename(columns={"name": "instructor_name"}),
        on="instructor_id", how="left"
    )
    if status is not None:
        info = info[info["status"] == status]
    if week_status is not None:
        info = info[info["week_status"] == week_status]
    if day_of_week is not None:
        info = info[info["day_of_week"] == day_of_week]

    info = info.sort_values(["class_date", "start_time", "booked_at"], ascending=[False, True, False])
    columns = ["booking_id", "class_id", "class_date", "day_of_week",
               "start_time", "end_time", "lesson_type", "instructor_name",
               "booked_at", "status", "week_status", "waitlist_position",
               "cancelled_at", "user_membership_id", "credit_deducted", "credit_restored"]
    records = info[columns].astype(object).where(pd.notna(info[columns]), None).to_dict("records")
    return {"success": True, "user_id": user_id,
            "user_name": user.iloc[0]["name"], "booking_count": len(records),
            "bookings": records,
            "message": "내 예약 내역 조회 완료" if records else "조건에 맞는 예약 내역이 없습니다."}
# 2-6. 약관 및 운영 정책 조회 함수

def search_policy_info(keyword=None):
    info = df_policies.copy()
    if keyword is not None:
        keyword = keyword.strip()
        if keyword:
            info = info[
                info["keyword"].astype(str).str.contains(keyword, case=False, regex=False, na=False) |
                info["content"].astype(str).str.contains(keyword, case=False, regex=False, na=False)
            ]
    if info.empty:
        return {"success": False, "policy_count": 0, "policies": [],
                "message": "조건에 해당하는 약관이 없습니다."}
    return {"success": True, "policy_count": len(info),
            "policies": info[["policy_id", "keyword", "content", "effective_date"]].to_dict("records")}

## **3.Process 설계**

In [6]:
# import json

# # 1. 실제 파이썬 함수 정의
# @tool
# def get_weather(location: str):
#     return f"{location}의 날씨는 맑음입니다."


# # 2. 사용할 함수들을 리스트(또는 툴 객체)로 묶기
# tools = [get_weather]
# # OpenAI나 LangChain이 함수 시그니처를 보고 JSON Schema를 자동 생성할 수 있음


# # 3. LLM에 함수를 바인딩(Binding)
# # 예: LangChain의 경우 llm.bind_tools(tools)
# model_with_tools = llm.bind_tools(tools)


# # 4. 사용자 질문으로 대화 시작
# messages = [
#     {"role": "user", "content": "서울 날씨 어때?"}
# ]

# response = model_with_tools.invoke(messages)


# # 5. LLM이 Tool Call을 반환하면 함수 실행
# if response.tool_calls:
#     messages.append(response)

#     for tool_call in response.tool_calls:
#         function_name = tool_call["name"]
#         function_args = tool_call["args"]

#         # 바인딩된 함수 매핑 테이블에서 실행
#         if function_name == "get_weather":
#             api_result = get_weather(**function_args)

#         # 6. 실행 결과를 다시 LLM에 전달
#         messages.append(
#             ToolMessage(
#                 content=api_result,
#                 tool_call_id=tool_call["id"]
#             )
#         )

#     # 7. 최종 답변 생성
#     final_response = model_with_tools.invoke(messages)
#     print(final_response.content)


### 시연 준비
- `PILATES_MODEL_REPO`를 로컬 **병합 모델 폴더** 또는 Hugging Face에 업로드한 병합 모델 ID로 설정하세요.
- Tool 데이터베이스 JSON은 원본 그대로 사용합니다.
- 6개 조회 함수, 회원 선택, 월·수 Multi-Call, 대화 맥락 및 Tool 결과 표시를 유지합니다.
- `gr.State`에는 직렬화 가능한 대화 내역만 저장합니다. vLLM 모델 객체를 보관하지 않습니다.
- Hugging Face 토큰·개인 데이터가 포함된 공개 Gradio 공유 링크는 데모 용도로만 사용하세요.

In [7]:
# 3. 병합된 Qwen 모델 vLLM 로딩 (LoRA 어댑터 경로가 아닌 병합 모델을 지정)
from pathlib import Path

MODEL_NAME = (os.getenv("PILATES_MODEL_REPO") or "").strip()
if not MODEL_NAME:
    if not HF_USERNAME:
        raise ValueError("PILATES_MODEL_REPO에 병합 모델 폴더 또는 Hub 저장소 ID를 설정하세요.")
    MODEL_NAME = f"{HF_USERNAME}/Qwen2.5-7B-Instruct-pilates-function-calling-v2"

# ADAPTER_PATH가 아니라 merge_and_unload()로 저장한 모델 폴더가 필요합니다
model_path = Path(MODEL_NAME).expanduser()
if model_path.exists() and not (model_path / "config.json").exists():
    raise ValueError(f"병합 모델 config.json이 없습니다: {model_path} (LoRA adapter 폴더인지 확인)")

if "llm" in globals():
    print("기존 vLLM 모델이 이미 로딩되어 있습니다. 재로딩이 필요하면 Kernel Restart 권장")
else:
    print("vLLM 모델 로딩:", MODEL_NAME)
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16",          # 양자화 없음
        gpu_memory_utilization=0.80,
        max_model_len=16384,
        max_num_seqs=1,
        enforce_eager=True,
    )

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=768,
    stop=["<|im_end|>", "<|endoftext|>"],
)
print("모델 준비 완료:", MODEL_NAME)

vLLM 모델 로딩: somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2
INFO 11-12 09:08:47 [utils.py:253] non-default args: {'dtype': 'float16', 'max_model_len': 16384, 'gpu_memory_utilization': 0.8, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2'}


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

INFO 11-12 09:08:59 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-12 09:08:59 [model.py:1745] Using max model len 16384
INFO 11-12 09:08:59 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 11-12 09:08:59 [vllm.py:500] Cudagraph is disabled under eager mode


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (9 outputs left out)

(EngineCore_DP0 pid=1566) INFO 11-12 09:10:51 [weight_utils.py:441] Time spent downloading weights for somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2: 90.966855 seconds


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1566) INFO 11-12 09:10:53 [default_loader.py:314] Loading weights took 2.55 seconds
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:54 [gpu_model_runner.py:3334] Model loading took 14.2488 GiB memory and 110.937927 seconds
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:56 [gpu_worker.py:359] Available KV cache memory: 48.02 GiB
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:56 [kv_cache_utils.py:1229] GPU KV cache size: 899,136 tokens
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:56 [kv_cache_utils.py:1234] Maximum concurrency for 16,384 tokens per request: 54.88x
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:56 [core.py:250] init engine (profile, create kv cache, warmup model) took 2.05 seconds
(EngineCore_DP0 pid=1566) INFO 11-12 09:10:57 [vllm.py:500] Cudagraph is disabled under eager mode
INFO 11-12 09:10:57 [llm.py:352] Supported tasks: ['generate']
모델 준비 완료: somsom317/Qwen2.5-7B-Instruct-pilates-function-calling-v2


In [16]:
import json
import re

class ConversationManager:
    def __init__(self, llm, sampling_params, user_id):
        self.history = []
        self.user_id = user_id
        self.system_prompt = r"""당신은 글로리 필라테스의 AI 상담사입니다. 성심성의껏 상담하십시오.

행동 판단 규칙:
- 이미 대화에서 제공된 정보 또는 일반적인 설명으로 답할 수 있으면 자연어로 바로 답합니다.
- 센터/회원 정보를 실제로 확인해야 하면 Tool을 호출합니다. 선택 파라미터 누락은 되물을 이유가 아닙니다.
- Tool로도 해결되지 않는 필수 정보의 모호함이나 서로 충돌하는 조건이 있을 때만 한 가지를 추가 질문합니다.
- 그 모호함이 해결되면 즉시 함수 호출 또는 자연어 답변을 이어갑니다.
- 서로 양립 가능한 복수 조건은 필요 시 여러 Tool을 호출합니다.
- 함수 호출 전 '확인해드리겠습니다' 같은 중간 안내 멘트는 출력하지 않습니다.
- 기존 결과로 답할 수 있으면 불필요한 재호출을 하지 않습니다.
- 제공하지 않는 예약 생성·취소·결제 작업은 완료했다고 주장하지 않습니다.

로그인한 사용자의 현재 ID: __LOGIN_USER_ID__
오늘 날짜: __REFERENCE_DATE__

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_membership_info", "description": "센터에 등록된 이용권 상품의 가격, 전체 이용 횟수, 유효기간, 구성 정보를 조회합니다. 전체 이용 횟수(total_sessions)는 sessions로 검색하며 SINGLE과 COMBO 모두 적용됩니다. SINGLE 상품은 lesson_type과 sessions로 검색할 수 있습니다. COMBO 상품은 product_type='COMBO'로 검색하고 반환된 components에서 구성별 레슨 유형과 횟수를 확인합니다. 콤보의 모든 구성과 횟수가 명시되어 정확한 전체 횟수를 알 수 있다면 합계를 sessions로 전달합니다. 일부 구성만 알려졌거나 특정 구성이 포함된 모든 상품을 찾을 때는 sessions를 생략합니다. 본인의 보유 이용권과 잔여 횟수는 get_my_memberships로 확인합니다. 조회 결과의 status를 확인하며, 반환된 모든 상품이 판매 중이라고 가정하지 않습니다.", "parameters": {"type": "object", "properties": {"sessions": {"type": "integer", "description": "상품의 전체 이용 횟수(total_sessions)에 정확히 일치하는 정수입니다. SINGLE과 COMBO 상품 모두에 적용됩니다. '6회권 있어?'는 sessions=6으로 검색합니다. '총 60회 콤보권 있어?'는 sessions=60, product_type='COMBO'로 검색합니다. 콤보 상품에서 전체 구성을 모두 알 수 있고 그 구성과 정확히 일치하는 상품을 찾는다면, 각 구성의 sessions를 합산하여 전달합니다. 예: '개인 6회 + 그룹 32회 콤보권'은 6+32=38이므로 sessions=38, product_type='COMBO'입니다. 예: '개인 12회 + 그룹 48회 콤보권'은 12+48=60이므로 sessions=60, product_type='COMBO'입니다. 단, total_sessions가 같아도 구성은 다를 수 있으므로 반환된 components를 반드시 검증합니다. '개인 12회가 포함된 콤보권'처럼 일부 구성만 지정했거나 다른 구성이 추가될 수 있는 '포함' 검색이면 전체 횟수를 확정할 수 없으므로 sessions를 생략하고 components로 판단합니다.(중요) '몇 회짜리가 있어?'처럼 횟수가 없는 경우에도 생략합니다. 존재 여부가 불확실한 횟수도 임의로 수정하지 않고 그대로 전달합니다. 구성별 개별 횟수(예: 6 또는 32)를 전체 횟수인 sessions로 잘못 전달하지 않습니다."}, "product_type": {"type": "string", "enum": ["SINGLE", "COMBO"], "description": "SINGLE은 한 종류의 레슨으로 구성된 단일 이용권, COMBO는 여러 레슨 유형을 결합한 이용권입니다. '단일권'을 명시하면 SINGLE을 선택하고, '개인+그룹', '듀엣+그룹', '복합', '혼합', '콤보'를 요청하면 COMBO를 선택합니다. 정확한 전체 횟수를 아는 COMBO 검색에서는 sessions를 함께 전달할 수 있습니다. 예: '개인 6회 + 그룹 32회 콤보권'은 product_type='COMBO', sessions=38입니다. COMBO의 구성별 횟수는 반환된 components에서 비교합니다. 특정 구성을 '포함'하는 상품은 해당 구성이 존재하는지 확인하고, '정확히 해당 구성'을 요구하면 추가 구성 없이 전체 구성이 일치하는지 확인합니다. COMBO 검색에는 lesson_type을 함께 전달하지 않습니다. 상품 유형이 지정되지 않으면 product_type을 생략하여 둘 다 검색할 수 있습니다. 단, lesson_type을 전달하면 데이터 구조상 그 레슨 유형의 SINGLE 상품만 조회됩니다."}, "lesson_type": {"type": "string", "enum": ["7:1 레슨", "2:1 듀엣 레슨", "1:1 개인 레슨"], "description": "SINGLE 상품의 레슨 유형을 정확히 검색합니다. '그룹', '그룹 수업', '그룹 레슨', '7:1'은 '7:1 레슨'으로, '듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 '2:1 듀엣 레슨'으로, '개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 개인 레슨'으로 변환합니다. '개인 12회 단일권'은 lesson_type='1:1 개인 레슨', sessions=12, product_type='SINGLE'로 검색합니다. COMBO는 수업 종류가 아니라 상품 구성 방식이므로 lesson_type에 COMBO를 넣지 않습니다. 콤보 상품의 구성 레슨은 product_type='COMBO'로 조회한 뒤 components에서 비교합니다. '개인 12회가 포함된 모든 상품'처럼 SINGLE과 COMBO를 모두 찾는 질문이면 SINGLE은 lesson_type과 sessions로, COMBO는 product_type='COMBO'로 각각 조회하고 components를 확인합니다. 레슨 유형이 지정되지 않으면 생략합니다. '7:1 듀엣'처럼 표현이 충돌하면 임의로 선택하지 않고 확인합니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_class_schedule", "description": "월~금 정규 수업 시간표를 조회합니다. '수업 언제 있어?', '강사 시간표 알려줘'처럼 운영 일정을 물을 때 사용합니다. 시간표는 정기 편성 정보이며 예약 잔여석을 확인하지 않습니다. '빈자리', '예약 가능'은 get_available_classes를 사용합니다. 월·수, 3시이후나 6시 이후처럼 여러 요일이나 요일별 다른 시간 조건은 각각 별도 호출합니다.", "parameters": {"type": "object", "properties": {"day_of_week": {"type": "string", "enum": ["월요일", "화요일", "수요일", "목요일", "금요일"], "description": "단일 요일만 입력합니다. '월·수'는 월요일, 수요일로 나누어 각각 호출합니다. 생략하면 월~금 전체를 조회합니다."}, "lesson_type": {"type": "string", "enum": ["7:1 레슨", "2:1 듀엣 레슨", "1:1 개인 레슨"], "description": "실제로 편성된 수업의 레슨 유형입니다. '그룹', '그룹 수업', '그룹 레슨', '7:1'은 '7:1 레슨', '듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 '2:1 듀엣 레슨', '개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 개인 레슨'으로 해석합니다. 콤보는 실제 수업 유형이 아니라 이용권의 구성 방식이므로 수업 lesson_type에 COMBO를 넣지 않습니다. 레슨 유형을 지정하지 않으면 전체 유형을 검색합니다. 서로 충돌하는 유형 표현(예: '7:1 듀엣')이 있으면 임의로 선택하지 않고 확인합니다."}, "instructor_name": {"type": "string", "description": "강사명 정확 일치 검색입니다. 사용자가 지정한 강사 이름을 전달하고, 이름이 불확실하면 확인합니다. 생략하면 전체 강사입니다."}, "start_time": {"type": "string", "description": "수업 시작 시각, HH:MM 형식입니다(예: 오후 3시='15:00'). time_filter와 함께 지정합니다."}, "end_time": {"type": "string", "description": "HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), 그 외에는 수업 자체 종료 시각의 상한(이하)입니다."}, "time_filter": {"type": "string", "enum": ["exact", "after", "at_or_after", "before", "at_or_before", "between"], "description": "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. 생략하면 exact입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_available_classes", "description": "예약이 오픈된 주차에서 실제 잔여석이 1석 이상인 수업만 조회합니다. '빈자리', '자리 남았어?', '예약 가능한 시간'일 때 사용합니다. 수업 예약 현황을 집계하므로 별도의 내 예약 조회 없이 일반 잔여석을 확인할 수 있지만, 회원의 이용권 사용 자격이나 같은 날짜 중복 예약 여부는 판단하지 않습니다. '내 이용권으로 김민지 선생님 수업 예약 가능?'이라면 get_my_memberships 및 get_my_bookings 결과와 함께 확인합니다. 만석 수업과 대기 명단은 반환하지 않습니다. 월·수처럼 서로 다른 요일 조건은 별도 호출합니다.", "parameters": {"type": "object", "properties": {"day_of_week": {"type": "string", "enum": ["월요일", "화요일", "수요일", "목요일", "금요일"], "description": "단일 요일만 입력합니다. '월·수'는 월요일, 수요일로 나누어 각각 호출합니다. 생략하면 월~금 전체를 조회합니다."}, "lesson_type": {"type": "string", "enum": ["7:1 레슨", "2:1 듀엣 레슨", "1:1 개인 레슨"], "description": "실제로 편성된 수업의 레슨 유형입니다. '그룹', '그룹 수업', '그룹 레슨', '7:1'은 '7:1 레슨', '듀엣', '듀엣 수업', '듀엣 레슨', '2:1'은 '2:1 듀엣 레슨', '개인', '개인 수업', '개인 레슨', '1:1'은 '1:1 개인 레슨'으로 해석합니다. 콤보는 실제 수업 유형이 아니라 이용권의 구성 방식이므로 수업 lesson_type에 COMBO를 넣지 않습니다. 레슨 유형을 지정하지 않으면 전체 유형을 검색합니다. 서로 충돌하는 유형 표현(예: '7:1 듀엣')이 있으면 임의로 선택하지 않고 확인합니다."}, "instructor_name": {"type": "string", "description": "강사명 정확 일치 검색입니다. 사용자가 지정한 강사 이름을 전달하고, 이름이 불확실하면 확인합니다. 생략하면 전체 강사입니다."}, "start_time": {"type": "string", "description": "수업 시작 시각, HH:MM 형식입니다(예: 오후 3시='15:00'). time_filter와 함께 지정합니다."}, "end_time": {"type": "string", "description": "HH:MM 형식입니다. between에서는 수업 시작 시각 검색 범위의 상한(양 끝 포함), 그 외에는 수업 자체 종료 시각의 상한(이하)입니다."}, "time_filter": {"type": "string", "enum": ["exact", "after", "at_or_after", "before", "at_or_before", "between"], "description": "수업의 시작 시각 기준: exact=정각 시작(==), before='이전' 미만(<), at_or_before='까지' 이하(<=), after='초과' 기준 시각 제외(>), at_or_after='부터' 이상(>=), between='A부터 B까지' 시작 시각 구간 양끝 포함입니다. 예: 3시 이전은 15:00 제외, 3시부터는 15:00 포함. 생략하면 exact입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_my_memberships", "description": "로그인한 회원 본인의 보유 이용권, 상품 유형, 유효기간, 남은 횟수를 조회합니다. 단일 및 콤보 이용권을 모두 반환하며 콤보 상품은 remaining_sessions_by_type에서 '7:1 레슨'(그룹), '2:1 듀엣 레슨'(듀엣), '1:1 개인 레슨'(개인)의 남은 횟수를 각각 확인합니다. 상품 구매 가능 목록은 get_membership_info, 실제 빈자리는 get_available_classes, 이미 예약한 날짜는 get_my_bookings로 확인합니다. 로그인 회원은 내부에서 결정하며 user_id 인자는 받지 않습니다.", "parameters": {"type": "object", "properties": {}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "get_my_bookings", "description": "로그인 회원 본인의 실제 수업 예약 날짜와 강사, 레슨 유형, 예약 상태를 조회합니다. 그룹=7:1 레슨, 듀엣=2:1 듀엣 레슨, 개인=1:1 개인 레슨으로 해석하며 콤보는 예약 수업 유형이 아닙니다. CONFIRMED=확정, WAITLISTED=대기, CANCELLED=취소, COMPLETED=완료를 구분합니다. 본인 예약 내역과 같은 날짜 중복 예약 여부를 확인할 때 사용하고, 전체 수업 빈자리는 get_available_classes로 조회합니다. 예약 생성·취소 기능은 없으며 user_id 인자는 받지 않습니다.", "parameters": {"type": "object", "properties": {"status": {"type": "string", "enum": ["CONFIRMED", "WAITLISTED", "CANCELLED", "COMPLETED"], "description": "예약 상태. CONFIRMED=예약 확정, WAITLISTED=예약 대기, CANCELLED=예약 취소, COMPLETED=수업 완료. 생략하면 전체 상태 조회."}, "week_status": {"type": "string", "enum": ["ACTIVE", "INACTIVE"], "description": "ACTIVE=현재 관리 중인 주차, INACTIVE=지난 주차. 생략하면 전체 주차 조회."}, "day_of_week": {"type": "string", "enum": ["월요일", "화요일", "수요일", "목요일", "금요일"], "description": "예약 내역의 단일 요일 필터입니다. '월·수'는 각각 호출합니다. 생략하면 전체 요일입니다."}}, "required": [], "additionalProperties": false}}}
{"type": "function", "function": {"name": "search_policy_info", "description": "센터의 예약 오픈, 취소, 대기, 출석, 환불 및 운영시간에 관한 정책과 약관을 조회합니다. '대기 예약 가능해?', '취소하면 횟수 돌려받아?'처럼 규정을 묻는 질문에 사용합니다. 실제 빈자리나 회원 본인의 예약 내역을 조회하는 함수는 아닙니다.", "parameters": {"type": "object", "properties": {"keyword": {"type": "string", "description": "정책 검색어(예: '예약', '취소', '대기', '출석', '환불', '운영')입니다. '대기 예약'이면 '대기'처럼 핵심어를 전달합니다. 생략하면 모든 정책을 조회합니다."}}, "required": [], "additionalProperties": false}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>"""

        # 회원 선택 결과와 고정 기준일을 System Prompt에 반영
        self.system_prompt = self.system_prompt.replace("__LOGIN_USER_ID__", user_id)
        self.system_prompt = self.system_prompt.replace("__REFERENCE_DATE__", REFERENCE_DATE)
        self.system_prompt += """

[실제 DB 응답 우선]
- 함수 결과에서 확인된 회원명, 이용권 횟수, 잔여 횟수, 강사명, 수업 시각과 ID만 그대로 사용합니다.
- 함수에 없는 값은 추측하거나 창작하지 말고 확인되지 않았다고 말합니다.
- 정규 시간표(get_class_schedule)와 예약 잔여석(get_available_classes)은 구분합니다.
- 월요일·수요일을 물어본 뒤 '제 이용권으로 가능한 수업'을 물으면 이전 요일 조건을 유지하고 이용권·빈자리 결과를 함께 확인합니다.
- 조회한 결과 중 일부만 전달했다면 전체를 안내한 것처럼 주장하지 않습니다.
- 여러 도구 호출 뒤에는 모든 도구 결과를 반영해 자연어로 답합니다.
"""

        self.llm = llm
        self.tokenizer = llm.get_tokenizer()  # Part2/3과 같은 Qwen Chat Template 사용
        self.sampling_params = sampling_params
        self.tool_call_pattern = re.compile(r'<tool_call>(.*?)</tool_call>', re.DOTALL)
        self.last_prompt = ""

    def format_conversation(self):
        """Part2 학습·Part3 평가와 같은 Qwen Chat Template (tool 역할 포함)."""
        messages = [{"role": "system", "content": self.system_prompt}, *self.history]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def extract_tool_call(self, assistant_message):
        """어시스턴트 응답에서 함수 호출 정보 추출"""
        match = self.tool_call_pattern.search(assistant_message)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                return None
        return None

    def add_message(self, role, content):
        """대화 이력에 메시지 추가"""
        self.history.append({"role": role, "content": content})

    def get_assistant_response(self, user_message):
        """사용자 메시지에 대한 어시스턴트 응답 생성"""
        self.add_message("user", user_message)
        prompt = self.format_conversation()
        self.last_prompt = prompt
        outputs = self.llm.generate(prompt, self.sampling_params)
        assistant_reply = outputs[0].outputs[0].text.strip()
        self.add_message("assistant", assistant_reply)
        return assistant_reply

    def process_function_result(self, function_result):
        """함수 결과를 학습 데이터처럼 tool 역할에 넣어 답변 생성."""
        temp_history = [*self.history, {"role": "tool", "content": str(function_result)}]
        prompt = self.tokenizer.apply_chat_template(
            [{"role": "system", "content": self.system_prompt}, *temp_history],
            tokenize=False, add_generation_prompt=True
        )
        self.last_prompt = prompt
        outputs = self.llm.generate(prompt, self.sampling_params)
        return outputs[0].outputs[0].text.strip()

import json
import re
import inspect
import threading
import gradio as gr


def create_interface(llm, sampling_params, max_tool_rounds=4):
    """세션별 대화/조회 기록을 안전하게 관리하는 Gradio 챗봇.

    gr.State에는 Python dict/list만 저장해 vLLM 인스턴스의 deepcopy 문제를 피함.
    """
    function_map = {
        "get_membership_info": get_membership_info,
        "get_class_schedule": get_class_schedule,
        "get_available_classes": get_available_classes,
        "get_my_memberships": get_my_memberships,
        "get_my_bookings": get_my_bookings,
        "search_policy_info": search_policy_info,
    }
    active_users = df_users.loc[df_users["status"] == "ACTIVE"]
    login_choices = [
        (f"{row['name']} ({row['user_id']})", str(row["user_id"]))
        for _, row in active_users.iterrows()
    ]
    if not login_choices:
        raise ValueError("ACTIVE 상태인 데모 회원이 없습니다. users.json 확인 필요")

    # vLLM.generate는 같은 LLM 객체에 대해 중복으로 호출하지 않게 직렬화
    generation_lock = threading.Lock()
    tokenizer = llm.get_tokenizer()
    tool_re = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)
    session_dict = lambda user_id: {"user_id": user_id, "history": []}

    def tool_calls(text):
        calls = []
        for match in tool_re.finditer(text or ""):
            try:
                item = json.loads(match.group(1))
                if isinstance(item, dict):
                    calls.append(item)
                else:
                    calls.append({"_parse_error": "함수 호출 JSON 객체가 아닙니다."})
            except (ValueError, TypeError) as exc:
                calls.append({"_parse_error": f"Tool JSON 오류: {exc}"})
        return calls

    def compact_response(result):
        """긴 DB 출력을 제한해 컨텍스트 초과를 방지 (원본 결과는 디버그창에 유지)."""
        max_rows = int(os.getenv("PILATES_MAX_TOOL_ROWS", "20"))
        max_rows = max(1, max_rows)
        if not isinstance(result, dict):
            return result
        result = dict(result)
        for k, v in list(result.items()):
            if isinstance(v, list) and len(v) > max_rows:
                result[k] = v[:max_rows]
                result["total_" + k] = len(v)
                result["truncated"] = True
                result["truncation_notice"] = (
                    "표시한 결과는 일부입니다. 나머지 항목의 시간·이름·ID는 추측하지 마세요."
                )
        return result

    def execute_tool(call, user_id):
        name = call.get("name") if isinstance(call, dict) else None
        args = call.get("arguments", {}) if isinstance(call, dict) else {}
        try:
            if not isinstance(call, dict) or "_parse_error" in call:
                raise ValueError((call or {}).get("_parse_error", "잘못된 함수 호출"))
            if name not in function_map:
                raise ValueError(f"허용되지 않은 함수: {name}")
            if not isinstance(args, dict):
                raise ValueError("arguments는 JSON 객체여야 합니다.")
            if "user_id" in args:
                raise ValueError("회원 ID는 선택된 로그인 세션으로만 결정합니다.")
            signature = inspect.signature(function_map[name])
            accepted = set(signature.parameters) - {"user_id"}
            unexpected = set(args) - accepted
            if unexpected:
                raise ValueError(f"지원되지 않는 인자: {sorted(unexpected)}")
            if name in ("get_my_memberships", "get_my_bookings"):
                result = function_map[name](**args, user_id=user_id)
            else:
                result = function_map[name](**args)
        except Exception as exc:
            result = {"success": False, "message": f"함수 호출 오류: {exc}"}
        return {"name": name, "arguments": args, "result": result}

    def generate(manager):
        prompt = manager.format_conversation()
        manager.last_prompt = prompt
        outputs = manager.llm.generate(prompt, manager.sampling_params)
        if not outputs or not outputs[0].outputs:
            raise RuntimeError("vLLM이 빈 출력을 반환했습니다.")
        text = outputs[0].outputs[0].text.strip()
        if not text:
            raise RuntimeError("모델 응답이 비어 있습니다.")
        return text

    def reset_session():
        return [], None, None, None, ""

    def respond(message, display_history, state, selected_user_id):
        display_history = list(display_history or [])
        message = (message or "").strip()
        if not selected_user_id:
            return "", display_history, state, None, None, "먼저 회원을 선택해주세요."
        if not message:
            return "", display_history, state, None, None, ""

        display_history.append({"role": "user", "content": message})
        if not isinstance(state, dict) or state.get("user_id") != selected_user_id:
            state = session_dict(selected_user_id)

        manager = ConversationManager(llm, sampling_params, selected_user_id)
        manager.history = list(state.get("history", []))
        all_calls, all_results = [], []

        try:
            with generation_lock:
                manager.add_message("user", message)
                reply = generate(manager)
                manager.add_message("assistant", reply)

                for _ in range(max_tool_rounds):
                    calls = tool_calls(reply)
                    if not calls:
                        break
                    all_calls.extend(calls)
                    batch = [execute_tool(call, selected_user_id) for call in calls]
                    all_results.extend(batch)
                    for item in batch:
                        # Qwen Chat Template은 tool role을 <tool_response>로 직렬화
                        compact = compact_response(item["result"])
                        manager.add_message(
                            "tool", json.dumps(compact, ensure_ascii=False, default=str)
                        )
                    reply = generate(manager)
                    manager.add_message("assistant", reply)

                if tool_calls(reply):
                    # 최대 횟수에 도달했으면 미완성 함수 호출을 그대로 보여주지 않음
                    reply = "조회가 반복되어 안내를 완료하지 못했습니다. 조건을 나누어 다시 질문해주세요."
                    manager.history[-1] = {"role": "assistant", "content": reply}

                # 모델이 Tool Call 외에 불필요한 구조 텍스트를 덧붙였으면 표시하지 않음
                if not reply.strip():
                    reply = "현재 답변을 생성하지 못했습니다. 다시 질문해주세요."
                display_history.append({"role": "assistant", "content": reply})
        except Exception as exc:
            # UI에서 예외를 확인하고 동일 입력을 다시 시도할 수 있게 유지
            reply = f"응답 생성 오류: {type(exc).__name__}: {exc}"
            display_history.append({"role": "assistant", "content": reply})
            # 실패한 모델 생성 시도의 user 메시지는 저장하지 않음
            manager.history = list(state.get("history", []))

        new_state = {"user_id": selected_user_id, "history": manager.history}
        return (
            "", display_history, new_state,
            all_calls or None, all_results or None,
            (manager.last_prompt[-12000:] if manager.last_prompt else "")
        )

    with gr.Blocks(title="글로리 필라테스 AI 상담사") as demo:
        gr.Markdown("# 글로리 필라테스 AI 상담사")
        gr.Markdown(
            f"시연 기준일: **{REFERENCE_DATE}** · 회원 선택은 데모 전용이며 실서비스 인증이 아닙니다."
        )
        login_user = gr.Dropdown(
            choices=login_choices, label="회원 선택 (데모)", value=None
        )
        # 순수 dict/list만 Gradio session state에 저장
        session_state = gr.State(value=None)
        chatbot = gr.Chatbot(height=520, type="messages", label="상담 대화")
        with gr.Row():
            user_input = gr.Textbox(
                placeholder="예: 월요일과 수요일 시간표 알려주세요",
                label="질문", scale=5
            )
            send_btn = gr.Button("전송", variant="primary", scale=1)
        clear_btn = gr.Button("대화 초기화")
        with gr.Accordion("개발 디버그 정보", open=False):
            with gr.Row():
                function_info = gr.JSON(label="함수 호출")
                function_result = gr.JSON(label="실제 DB 반환 결과")
            prompt_debug = gr.Textbox(label="마지막 모델 프롬프트 (일부)", lines=6)

        login_user.change(
            fn=reset_session,
            inputs=[],
            outputs=[chatbot, session_state, function_info, function_result, prompt_debug],
            queue=False,
        )
        clear_btn.click(
            fn=reset_session,
            inputs=[],
            outputs=[chatbot, session_state, function_info, function_result, prompt_debug],
            queue=False,
        )
        inputs = [user_input, chatbot, session_state, login_user]
        outputs = [user_input, chatbot, session_state, function_info, function_result, prompt_debug]
        send_btn.click(respond, inputs=inputs, outputs=outputs, concurrency_limit=1)
        user_input.submit(respond, inputs=inputs, outputs=outputs, concurrency_limit=1)

    return demo


# RunPod 프록시 이용 시 share=False로 두고 7860 포트를 사용하거나
# 외부 데모 링크가 필요하면 PILATES_SHARE=true로 설정합니다
demo = create_interface(llm, sampling_params)
DEMO_SHARE = os.getenv("PILATES_SHARE", "false").lower() == "true"
print("Gradio 버전:", gr.__version__)
print("공유 링크 사용:", DEMO_SHARE)
demo.queue(default_concurrency_limit=1)

import socket

# 사용 가능한 포트 자동 선택
def find_free_port(start=7861, end=7900):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    raise RuntimeError("사용 가능한 포트가 없습니다.")

PORT = find_free_port()

print("Gradio 버전:", gr.__version__)
print(f"Gradio 실행 포트: {PORT}")

demo.queue(default_concurrency_limit=1)

demo.launch(
    server_name="0.0.0.0",
    server_port=PORT,
    share=True,
    show_error=True
)

/tmp/ipykernel_978/331916706.py:278: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=520, type="messages", label="상담 대화")


Gradio 버전: 5.50.0
공유 링크 사용: False
Gradio 버전: 5.50.0
Gradio 실행 포트: 7864


[2026-11-12 09:17:08] INFO _client.py:1025: HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
[2026-11-12 09:17:08] INFO _client.py:1025: HTTP Request: GET http://localhost:7864/gradio_api/startup-events "HTTP/1.1 200 OK"
[2026-11-12 09:17:08] INFO _client.py:1025: HTTP Request: HEAD http://localhost:7864/ "HTTP/1.1 200 OK"


* Running on local URL:  http://0.0.0.0:7864


[2026-11-12 09:17:08] INFO _client.py:1025: HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"
[2026-11-12 09:17:08] INFO _client.py:1025: HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"


* Running on public URL: https://23bb17dc2dd8ff8cc6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


... (3 outputs left out)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]